**Name:- Shashiraj Y Rao**<br>
**Roll No:- IMT2024078**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore")

# ---------------- Settings ----------------
ID = 'IMT2024078'
CV_FOLDS = 5
MAX_DEGREE = 20
DEGREES = list(range(1, MAX_DEGREE + 1))
TOL = 0.001            # simplest model within this much CV R^2 of the best
FORCE_DEGREE = None    # e.g. 11 to force degree 11 for every model; None = auto
WEAK_ALPHA = 1e-6      # near-unregularized, used to expose overfitting

# ---------------- Data (var2) ----------------
df = pd.read_csv(f'{ID}_train_var2(in).csv')
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_unseen = pd.read_csv(f'{ID}_test_var2(in).csv').values

# ---------------- Models ----------------
models = {
    'OLS (no reg.)': (LinearRegression(), {}),
    'Ridge':         (Ridge(), {'alpha': [0.1, 1.0, 10.0]}),
    'Lasso':         (Lasso(max_iter=10000), {'alpha': [0.1, 1.0, 10.0]}),
    'ElasticNet':    (ElasticNet(max_iter=10000),
                      {'alpha': [0.1, 1.0, 10.0], 'l1_ratio': [0.2, 0.5, 0.8]}),
}
weak_models = {
    f'Ridge (weak, α={WEAK_ALPHA:g})':      Ridge(alpha=WEAK_ALPHA),
    f'Lasso (weak, α={WEAK_ALPHA:g})':      Lasso(alpha=WEAK_ALPHA, max_iter=10000),
    f'ElasticNet (weak, α={WEAK_ALPHA:g})': ElasticNet(alpha=WEAK_ALPHA, l1_ratio=0.5, max_iter=10000),
}


def select_model(cv_results):
    """Refit rule: forced degree, or simplest degree within TOL of best CV R^2."""
    d = pd.DataFrame(cv_results)
    d['deg'] = d['param_poly__degree'].astype(int)
    d['alp'] = d['param_model__alpha'].astype(float) if 'param_model__alpha' in d else 0.0
    if FORCE_DEGREE is not None:
        return int(d[d['deg'] == FORCE_DEGREE]['mean_test_r2'].idxmax())
    best = d['mean_test_r2'].max()
    cand = d[d['mean_test_r2'] >= best - TOL].sort_values(['deg', 'alp'], ascending=[True, False])
    return int(cand.index[0])


# ---------------- Tuned models ----------------
fitted, curves = {}, {}
for name, (est, grid) in models.items():
    print(f"Fitting {name} ...")
    pg = {'poly__degree': DEGREES}
    pg.update({f'model__{k}': v for k, v in grid.items()})
    gs = GridSearchCV(
        Pipeline([('poly', PolynomialFeatures()), ('model', est)]),
        pg, cv=CV_FOLDS,
        scoring={'r2': 'r2', 'neg_mse': 'neg_mean_squared_error'},
        refit=select_model, n_jobs=-1)
    gs.fit(X_train, y_train)
    fitted[name] = gs
    res = pd.DataFrame(gs.cv_results_)
    b = res.loc[res.groupby('param_poly__degree')['mean_test_r2'].idxmax()]
    curves[name] = (b['param_poly__degree'].astype(int).values,
                    b['mean_test_r2'].values,
                    -b['mean_test_neg_mse'].values)

# ---------------- Weak-regularization curves ----------------
for name, est in weak_models.items():
    print(f"Evaluating {name} ...")
    r2s, mses = [], []
    for d in DEGREES:
        pipe = Pipeline([('poly', PolynomialFeatures(degree=d)), ('model', est)])
        cv = cross_validate(pipe, X_train, y_train, cv=CV_FOLDS,
                            scoring={'r2': 'r2', 'neg_mse': 'neg_mean_squared_error'}, n_jobs=-1)
        r2s.append(cv['test_r2'].mean())
        mses.append(-cv['test_neg_mse'].mean())
    curves[name] = (np.array(DEGREES), np.array(r2s), np.array(mses))

# ---------------- Single combined figure ----------------
w = f'(weak, α={WEAK_ALPHA:g})'
styles = {
    'OLS (no reg.)': ('black', 'o', '-'),
    'Ridge': ('tab:blue', 's', '-'),
    'Lasso': ('tab:green', '^', '-'),
    'ElasticNet': ('tab:red', 'D', '-'),
    f'Ridge {w}': ('tab:blue', 's', '--'),
    f'Lasso {w}': ('tab:green', '^', '--'),
    f'ElasticNet {w}': ('tab:red', 'D', '--'),
}
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
for name, (deg, r2, mse) in curves.items():
    c, m, ls = styles[name]
    ax[0].plot(deg, r2, color=c, marker=m, linestyle=ls, linewidth=2, label=name)
    ax[1].plot(deg, mse, color=c, marker=m, linestyle=ls, linewidth=2, label=name)

ax[0].set_title('Polynomial Degree vs CV $R^2$ (var2)')
ax[0].set_ylabel('Mean CV $R^2$')
ax[0].set_ylim(bottom=max(-1, min(v[1].min() for v in curves.values()) - 0.05), top=1.02)
ax[1].set_title('Polynomial Degree vs CV MSE (var2)')
ax[1].set_ylabel('Mean CV MSE (log scale)')
ax[1].set_yscale('log')
for a in ax:
    a.set_xlabel('Polynomial Degree')
    a.set_xticks(range(1, MAX_DEGREE + 1, 1))
    a.grid(True, linestyle='--', alpha=0.6)
    a.legend(fontsize=8)
plt.tight_layout()
plt.savefig('var2_degree_vs_r2_mse_all_models.png', dpi=150)
plt.show()

# ---------------- Comparison on held-out test split ----------------
rows = []
for name, gs in fitted.items():
    pred = gs.best_estimator_.predict(X_test)
    rows.append({
        'Model': name,
        'Degree': gs.best_params_['poly__degree'],
        'Params': {k.replace('model__', ''): v for k, v in gs.best_params_.items() if k.startswith('model__')},
        'CV R2': round(pd.DataFrame(gs.cv_results_).loc[gs.best_index_, 'mean_test_r2'], 4),
        'Test R2': round(r2_score(y_test, pred), 4),
        'Test MSE': round(mean_squared_error(y_test, pred), 6),
    })
summary = pd.DataFrame(rows).sort_values('Test MSE')
print("\n===== Model comparison (held-out test split) =====")
print(summary.to_string(index=False))

winner = summary.iloc[0]['Model']
print(f"\nBetter model (lowest test MSE / highest test R2): {winner}")

# ---------------- Predictions from the winner ----------------
final_pred = fitted[winner].best_estimator_.predict(X_unseen)
pd.DataFrame({'Predicted_Thermal_Anomaly_Score': final_pred}).to_csv(
    f'{ID}_predictions_best.csv', index=False)
print(f"Saved predictions from {winner} to '{ID}_predictions_best.csv'")